<a href="https://colab.research.google.com/github/ultimate-kiran/pdf_chatbot/blob/main/pdf_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install all required libraries
!pip install langchain
!pip install langchain-community
!pip install langchain-google-genai
!pip install google-generativeai
!pip install faiss-cpu
!pip install pypdf
!pip install gradio
!pip install sentence-transformers
!pip install chromadb
!pip install tiktoken

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [ ]:
# Install the missing package
!pip install langchain-text-splitters -q

print("✅ Installed successfully!")

✅ Installed successfully!


In [ ]:
!pip install langchain-groq -q
print("✅ Groq Installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00
✅ Groq Installed!


In [ ]:
from google.colab import userdata
import os

# Load from Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Groq API Key Loaded!")

✅ Groq API Key Loaded!


In [ ]:
# PDF Upload and Text Extraction - FIXED VERSION
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ Fixed import
import tempfile
import os

def upload_and_extract_pdf():
    """Upload PDF and extract text"""
    print("📁 Please upload your PDF file...")

    # Upload file
    uploaded = files.upload()

    documents = []

    for filename, content in uploaded.items():
        print(f"📄 Processing: {filename}")

        # Save to temp file
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix='.pdf'
        ) as tmp_file:
            tmp_file.write(content)
            tmp_path = tmp_file.name

        # Load PDF
        loader = PyPDFLoader(tmp_path)
        docs = loader.load()
        documents.extend(docs)

        print(f"✅ Extracted {len(docs)} pages from {filename}")

        # Clean up
        os.unlink(tmp_path)

    return documents

# Run PDF upload
raw_documents = upload_and_extract_pdf()
print(f"\n📊 Total pages extracted: {len(raw_documents)}")

📁 Please upload your PDF file...


KeyboardInterrupt: 

In [ ]:
# Split text into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunks(documents):
    """Split documents into chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # Each chunk = 1000 characters
        chunk_overlap=200,    # Overlap to avoid losing context
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    print(f"📊 Total chunks created: {len(chunks)}")
    print(f"📝 Sample chunk preview:")
    print("-" * 50)
    print(chunks[0].page_content[:300])
    print("-" * 50)

    return chunks

# Create chunks
chunks = create_chunks(raw_documents)

📊 Total chunks created: 14
📝 Sample chunk preview:
--------------------------------------------------
CHAPTER 7 
CONCLUSION AND FUTURE WORK 
7.1  CONCLUSION 
In this paper, a deep learning based  framework for Text –Pedestrian Image 
Retrieval (TPIR) has been presented to solve the problematic issue of finding images of 
pedestrians through natural language descriptions. The main problem with TPIR i
--------------------------------------------------


In [ ]:
# STEP 6 - Using Groq (No Daily Limits!)
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def setup_rag_chain_groq(vector_store):

    print("🤖 Setting up Groq LLM...")

    # ✅ Groq - FREE with no daily limits
    llm = ChatGroq(
        model="llama-3.1-8b-instant",  # Fast and free
        temperature=0.3,
        max_tokens=2048
    )

    print("✅ Groq LLM Ready!")

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

    print("✅ Retriever Ready!")

    prompt_template = PromptTemplate.from_template("""
    You are a helpful AI Study Assistant and Personal Tutor.

    Use the context from the student notes to answer
    the question accurately and clearly.

    Context from Notes:
    {context}

    Student Question:
    {question}

    Instructions:
    - Answer clearly in simple language
    - Use bullet points if helpful
    - Use examples if needed
    - If not in context say: "This topic is not in your notes"
    - Be encouraging and supportive

    Answer:
    """)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt_template
        | llm
        | StrOutputParser()
    )

    print("✅ RAG Chain Ready!")
    print("\n🚀 Pipeline:")
    print("   📄 Notes → 🧠 FAISS → 🤖 Groq Llama → 💬 Answer")
    print("   ✅ No daily limits! Use freely!\n")

    return rag_chain, llm, retriever


# Setup Groq RAG
rag_chain, llm, retriever = setup_rag_chain_groq(vector_store)

🤖 Setting up Groq LLM...
✅ Groq LLM Ready!
✅ Retriever Ready!
✅ RAG Chain Ready!

🚀 Pipeline:
   📄 Notes → 🧠 FAISS → 🤖 Groq Llama → 💬 Answer
   ✅ No daily limits! Use freely!



In [ ]:
# Test Question
ask_question(
    "What are the main topics in these notes?",
    rag_chain
)


🙋 Question: What are the main topics in these notes?
🤔 Thinking...

💡 Answer:
It looks like you've been working hard on your notes. I'm happy to help you summarize the main topics.

Here are the main topics from your notes:

* **Text-Pedestrian Image Retrieval (TPIR)**: This is the main topic of your notes. TPIR is a deep learning-based framework that helps find images of pedestrians through natural language descriptions.
* **Semantic Gap**: The semantic gap is the difficulty in overcoming the difference between text and visual image representations.
* **Transformer Architecture**: Your notes mention the use of transformer-based architecture, which allows the model to record long-range dependencies in text and images.
* **Progressive Feature Mining (PFM)**: PFM is a module that helps refine features and suppress dominant responding while emphasizing complimentary identity-discriminative information.
* **Graph Attention Module (GAM)**: GAM is used to explain semantic interconnections a

"It looks like you've been working hard on your notes. I'm happy to help you summarize the main topics.\n\nHere are the main topics from your notes:\n\n* **Text-Pedestrian Image Retrieval (TPIR)**: This is the main topic of your notes. TPIR is a deep learning-based framework that helps find images of pedestrians through natural language descriptions.\n* **Semantic Gap**: The semantic gap is the difficulty in overcoming the difference between text and visual image representations.\n* **Transformer Architecture**: Your notes mention the use of transformer-based architecture, which allows the model to record long-range dependencies in text and images.\n* **Progressive Feature Mining (PFM)**: PFM is a module that helps refine features and suppress dominant responding while emphasizing complimentary identity-discriminative information.\n* **Graph Attention Module (GAM)**: GAM is used to explain semantic interconnections among textual tokens and help the model learn to be sensitive to contex

In [ ]:
# Test Summary
summarize_notes(raw_documents, llm)


📋 Generating Summary of Your Notes...

📋 SUMMARY:
**📌 Main Topic**
Text-Pedestrian Image Retrieval (TPIR) using Deep Learning

**🔑 Key Concepts**

* Text-Pedestrian Image Retrieval (TPIR): finding images of pedestrians through natural language descriptions
* Semantic gap: difficulty in overcoming the difference between text and visual image representations
* Feature extraction using transformers
* Progressive feature refinement
* Graph-based feature refinement
* BERT model for contextual semantic representations
* ViT architecture for global image representations of pedestrians
* Progressive Feature Mining (PFM) module
* Dual Reverse Attention Module (DRAM)
* Single Reverse Attention Module (SRAM)
* Graph Attention Module (GAM)
* Cross-modal alignment and retrieval functioning

**📊 Important Points to Remember**

* The proposed framework combines feature extraction, progressive feature refinement, and graph-based feature refinement to address the semantic gap in TPIR.
* The framework 

'**📌 Main Topic**\nText-Pedestrian Image Retrieval (TPIR) using Deep Learning\n\n**🔑 Key Concepts**\n\n* Text-Pedestrian Image Retrieval (TPIR): finding images of pedestrians through natural language descriptions\n* Semantic gap: difficulty in overcoming the difference between text and visual image representations\n* Feature extraction using transformers\n* Progressive feature refinement\n* Graph-based feature refinement\n* BERT model for contextual semantic representations\n* ViT architecture for global image representations of pedestrians\n* Progressive Feature Mining (PFM) module\n* Dual Reverse Attention Module (DRAM)\n* Single Reverse Attention Module (SRAM)\n* Graph Attention Module (GAM)\n* Cross-modal alignment and retrieval functioning\n\n**📊 Important Points to Remember**\n\n* The proposed framework combines feature extraction, progressive feature refinement, and graph-based feature refinement to address the semantic gap in TPIR.\n* The framework uses BERT and ViT architectur

In [ ]:
# Test Quiz
generate_quiz(raw_documents, llm, num_questions=3)


🧪 Generating 3 Quiz Questions...

🧪 QUIZ:
Q1: What is the main problem that the proposed framework in the paper aims to solve in Text-Pedestrian Image Retrieval (TPIR)?

A) Difficulty in extracting features from images
B) Difficulty in overcoming the semantic gap between text and visual image representations
C) Difficulty in training the model on large datasets
D) Difficulty in retrieving images from a gallery database

✅ Correct Answer: B) Difficulty in overcoming the semantic gap between text and visual image representations - The paper highlights the difficulty in overcoming the semantic gap between text and visual image representations as the main problem in TPIR.

---

Q2: What is the purpose of the Progressive Feature Mining (PFM) module in the proposed framework?

A) To extract features from images using transformers
B) To refine multimodal features and suppress dominant responding
C) To introduce a Graph Attention Module (GAM) to explain semantic interconnections among textual

'Q1: What is the main problem that the proposed framework in the paper aims to solve in Text-Pedestrian Image Retrieval (TPIR)?\n\nA) Difficulty in extracting features from images\nB) Difficulty in overcoming the semantic gap between text and visual image representations\nC) Difficulty in training the model on large datasets\nD) Difficulty in retrieving images from a gallery database\n\n✅ Correct Answer: B) Difficulty in overcoming the semantic gap between text and visual image representations - The paper highlights the difficulty in overcoming the semantic gap between text and visual image representations as the main problem in TPIR.\n\n---\n\nQ2: What is the purpose of the Progressive Feature Mining (PFM) module in the proposed framework?\n\nA) To extract features from images using transformers\nB) To refine multimodal features and suppress dominant responding\nC) To introduce a Graph Attention Module (GAM) to explain semantic interconnections among textual tokens\nD) To map refined 

In [ ]:
# Install huggingface
!pip install sentence-transformers -q

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def create_vector_store_huggingface(chunks):
    """Use FREE HuggingFace embeddings - No API key needed!"""

    print("🤗 Loading HuggingFace embedding model...")
    print("⏳ First time download may take 1-2 minutes...")

    # FREE local embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

    print("✅ HuggingFace model loaded!")

    # Test
    test = embeddings.embed_query("test")
    print(f"📐 Vector size: {len(test)}")

    # Create vector store
    print("📦 Creating FAISS vector store...")
    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    print("✅ Vector store ready!")
    print(f"📦 Vectors stored: {vector_store.index.ntotal}")

    return vector_store, embeddings

# Use this if Gemini fails
vector_store, embeddings = create_vector_store_huggingface(chunks)

🤗 Loading HuggingFace embedding model...
⏳ First time download may take 1-2 minutes...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ HuggingFace model loaded!
📐 Vector size: 384
📦 Creating FAISS vector store...
✅ Vector store ready!
📦 Vectors stored: 14


In [ ]:
# Run this first
!pip install langchain -q
!pip install langchain-core -q

print("✅ Done!")

✅ Done!


In [ ]:
# STEP 6 - Setup Gemini LLM & RAG Chain (FULLY FIXED VERSION)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def setup_rag_chain(vector_store):
    """Setup RAG chain with Gemini - Modern LangChain way"""

    print("🤖 Setting up Gemini LLM...")

    # ✅ Initialize Gemini LLM
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        temperature=0.3,
        max_output_tokens=2048
    )

    print("✅ Gemini LLM Ready!")

    # ✅ Create Retriever
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

    print("✅ Retriever Ready!")

    # ✅ Custom Prompt Template
    prompt_template = PromptTemplate.from_template("""
    You are a helpful AI Study Assistant and Personal Tutor.

    Use the following context from the student's notes/PDF to answer
    the question accurately and clearly.

    Context from Notes:
    {context}

    Student's Question: {question}

    Instructions:
    - Answer clearly and in simple language
    - Use examples if needed
    - If answer is not in context, say "This topic is not covered in your notes"
    - Be encouraging and supportive as a tutor

    Answer:
    """)

    print("✅ Prompt Template Ready!")

    # ✅ Helper function to format documents
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # ✅ Build Modern RAG Chain (LCEL style)
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt_template
        | llm
        | StrOutputParser()
    )

    print("✅ RAG Chain Ready!")
    print("\n🚀 Everything Connected:")
    print("   📄 Your Notes → 🧠 FAISS → 🤖 Gemini LLM")
    print("   ✅ AI Study Assistant is READY!\n")

    return rag_chain, llm, retriever

# Run Setup
rag_chain, llm, retriever = setup_rag_chain(vector_store)

🤖 Setting up Gemini LLM...
✅ Gemini LLM Ready!
✅ Retriever Ready!
✅ Prompt Template Ready!
✅ RAG Chain Ready!

🚀 Everything Connected:
   📄 Your Notes → 🧠 FAISS → 🤖 Gemini LLM
   ✅ AI Study Assistant is READY!



In [ ]:
# STEP 7 - All Features (FULLY FIXED)
from langchain_core.messages import HumanMessage  # ✅ Fixed import

# ============================================
# FEATURE 1: Ask Questions (Q&A)
# ============================================
def ask_question(question, rag_chain):
    """Ask a question about your notes"""

    print("\n" + "="*60)
    print(f"🙋 Question: {question}")
    print("="*60)
    print("🤔 Thinking...")

    answer = rag_chain.invoke(question)

    print(f"\n💡 Answer:\n{answer}")
    print("="*60)

    return answer


# ============================================
# FEATURE 2: Summarize Notes
# ============================================
def summarize_notes(raw_documents, llm):
    """Generate summary of uploaded notes"""

    print("\n" + "="*60)
    print("📋 Generating Summary of Your Notes...")
    print("="*60)

    # Get text
    full_text = " ".join([doc.page_content for doc in raw_documents])
    sample_text = full_text[:3000]

    prompt = f"""
    Please provide a comprehensive summary of these study notes.

    Notes:
    {sample_text}

    Provide:
    1. 📌 Main Topic
    2. 🔑 Key Concepts (bullet points)
    3. 📊 Important Points to Remember
    4. 🎯 What to Focus On

    Keep it clear and student-friendly.
    """

    # ✅ Fixed - using langchain_core
    response = llm.invoke([HumanMessage(content=prompt)])

    print("\n📋 SUMMARY:")
    print(response.content)
    print("="*60)

    return response.content


# ============================================
# FEATURE 3: Generate Quiz (MCQs)
# ============================================
def generate_quiz(raw_documents, llm, num_questions=5):
    """Generate MCQ quiz from notes"""

    print("\n" + "="*60)
    print(f"🧪 Generating {num_questions} Quiz Questions...")
    print("="*60)

    full_text = " ".join([doc.page_content for doc in raw_documents])
    sample_text = full_text[:3000]

    prompt = f"""
    Create {num_questions} Multiple Choice Questions from these notes.

    Notes:
    {sample_text}

    Format exactly like this:

    Q1: [Question]
    A) [Option 1]
    B) [Option 2]
    C) [Option 3]
    D) [Option 4]
    ✅ Correct Answer: [A/B/C/D] - [Explanation]

    ---

    Make questions that test real understanding.
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    print("\n🧪 QUIZ:")
    print(response.content)
    print("="*60)

    return response.content


# ============================================
# FEATURE 4: Explain Topic Simply
# ============================================
def explain_topic(topic, rag_chain):
    """Explain any topic simply"""

    print("\n" + "="*60)
    print(f"🎓 Explaining: {topic}")
    print("="*60)

    question = f"""
    Explain '{topic}' in very simple words with:
    - Simple language a beginner understands
    - A real life example
    - Key points in bullet format
    """

    answer = rag_chain.invoke(question)

    print("\n💡 Simple Explanation:")
    print(answer)
    print("="*60)

    return answer


# ============================================
# FEATURE 5: Generate Flashcards
# ============================================
def generate_flashcards(raw_documents, llm, num_cards=5):
    """Generate flashcards for revision"""

    print("\n" + "="*60)
    print(f"🃏 Generating {num_cards} Flashcards...")
    print("="*60)

    full_text = " ".join([doc.page_content for doc in raw_documents])
    sample_text = full_text[:3000]

    prompt = f"""
    Create {num_cards} Flashcards from these study notes.

    Notes:
    {sample_text}

    Format exactly like this:

    🃏 CARD 1:
    FRONT: [Question or Term]
    BACK:  [Answer or Definition]

    ---

    Make cards useful for quick revision.
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    print("\n🃏 FLASHCARDS:")
    print(response.content)
    print("="*60)

    return response.content


print("✅ All 5 Features Ready!")
print("\n📌 Features Available:")
print("   1️⃣  ask_question()          - Ask from notes")
print("   2️⃣  summarize_notes()       - Get summary")
print("   3️⃣  generate_quiz()         - MCQ Quiz")
print("   4️⃣  explain_topic()         - Simple explanation")
print("   5️⃣  generate_flashcards()   - Revision cards")

✅ All 5 Features Ready!

📌 Features Available:
   1️⃣  ask_question()          - Ask from notes
   2️⃣  summarize_notes()       - Get summary
   3️⃣  generate_quiz()         - MCQ Quiz
   4️⃣  explain_topic()         - Simple explanation
   5️⃣  generate_flashcards()   - Revision cards


In [ ]:
# STEP 6 - Fixed with gemini-pro
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def setup_rag_chain(vector_store):
    """Setup RAG chain with gemini-pro"""

    print("🤖 Setting up Gemini Pro LLM...")

    # ✅ Use gemini-pro
    llm = ChatGoogleGenerativeAI(
        model="gemini-pro",
        temperature=0.3,
        max_output_tokens=2048,
        convert_system_message_to_human=True  # ✅ Required for gemini-pro
    )

    print("✅ Gemini Pro LLM Ready!")

    # ✅ Create Retriever
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )
    print("✅ Retriever Ready!")

    # ✅ Prompt Template
    prompt_template = PromptTemplate.from_template("""
    You are a helpful AI Study Assistant and Personal Tutor.

    Use the following context from the student notes to answer
    the question accurately and clearly.

    Context from Notes:
    {context}

    Student Question: {question}

    Instructions:
    - Answer clearly and in simple language
    - Use examples if needed
    - If answer is not in context say "This topic is not in your notes"
    - Be encouraging and supportive

    Answer:
    """)

    print("✅ Prompt Template Ready!")

    # ✅ Format documents helper
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # ✅ Build RAG Chain
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt_template
        | llm
        | StrOutputParser()
    )

    print("✅ RAG Chain Ready!")
    print("\n🚀 Pipeline Connected:")
    print("   📄 Notes → 🧠 FAISS → 🤖 Gemini Pro → 💬 Answer")
    print("   ✅ AI Study Assistant is READY!\n")

    return rag_chain, llm, retriever

# Run setup
rag_chain, llm, retriever = setup_rag_chain(vector_store)

🤖 Setting up Gemini Pro LLM...
✅ Gemini Pro LLM Ready!
✅ Retriever Ready!
✅ Prompt Template Ready!
✅ RAG Chain Ready!

🚀 Pipeline Connected:
   📄 Notes → 🧠 FAISS → 🤖 Gemini Pro → 💬 Answer
   ✅ AI Study Assistant is READY!



In [ ]:
# Install Gradio
!pip install gradio -q
print("✅ Gradio Installed!")

✅ Gradio Installed!


In [ ]:
# STEP 10 - Beautiful Gradio Web UI
import gradio as gr
from langchain_core.messages import HumanMessage

# ============================================
# Helper Functions for Gradio
# ============================================

def chat_fn(question, history):
    """Chat function"""
    if not question.strip():
        return "⚠️ Please type a question!"
    try:
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        return f"❌ Error: {str(e)}"


def summary_fn():
    """Summary function"""
    try:
        full_text = " ".join([doc.page_content for doc in raw_documents])
        sample_text = full_text[:3000]

        prompt = f"""
        Summarize these study notes clearly.
        Notes: {sample_text}

        Give:
        1. 📌 Main Topic
        2. 🔑 Key Concepts (bullets)
        3. 📊 Important Points
        4. 🎯 What to Focus On
        """
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        return f"❌ Error: {str(e)}"


def quiz_fn(num_questions):
    """Quiz generator function"""
    try:
        full_text = " ".join([doc.page_content for doc in raw_documents])
        sample_text = full_text[:3000]

        prompt = f"""
        Create {int(num_questions)} MCQ questions from these notes.
        Notes: {sample_text}

        Format:
        Q1: [Question]
        A) [Option]
        B) [Option]
        C) [Option]
        D) [Option]
        Correct Answer: [A/B/C/D] - [Explanation]
        ---
        """
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        return f"❌ Error: {str(e)}"


def flashcard_fn(num_cards):
    """Flashcard generator"""
    try:
        full_text = " ".join([doc.page_content for doc in raw_documents])
        sample_text = full_text[:3000]

        prompt = f"""
        Create {int(num_cards)} flashcards from these notes.
        Notes: {sample_text}

        Format:
        🃏 CARD 1:
        FRONT: [Term or Question]
        BACK: [Definition or Answer]
        ---
        """
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        return f"❌ Error: {str(e)}"


def explain_fn(topic):
    """Explain topic function"""
    if not topic.strip():
        return "⚠️ Please enter a topic!"
    try:
        question = f"Explain {topic} simply with examples and key points"
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        return f"❌ Error: {str(e)}"


print("✅ All Gradio functions ready!")

✅ All Gradio functions ready!


In [ ]:
!pip install langchain-groq -q
print("✅ Done!")

✅ Done!


In [ ]:
import gradio as gr
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_community.document_loaders import PyPDFLoader
from google.colab import userdata
import os

# ============================================
# LOAD GROQ API KEY
# ============================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ Groq API Key Loaded!")

# ============================================
# GLOBAL VARIABLES
# ============================================
rag_chain = None
llm = None
raw_documents = None


# ============================================
# PROCESS UPLOADED FILE
# ============================================
def process_file(file):
    global rag_chain, llm, raw_documents

    if file is None:
        return "⚠️ Please upload a PDF file first!"

    try:
        # Load PDF
        loader = PyPDFLoader(file.name)
        raw_documents = loader.load()

        # Split
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )
        chunks = splitter.split_documents(raw_documents)

        # Embeddings
        embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2"
        )

        # Vector Store
        vector_store = FAISS.from_documents(chunks, embeddings)
        retriever = vector_store.as_retriever(search_kwargs={"k": 3})

        # ✅ Groq LLM
        llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.3,
            max_tokens=2048
        )

        # Prompt
        prompt = PromptTemplate.from_template("""
        You are a helpful AI Study Assistant and Personal Tutor.
        Use the context from the notes to answer clearly.

        Context:
        {context}

        Question:
        {question}

        Answer clearly with bullet points:
        """)

        def format_docs(docs):
            return "\n\n".join(d.page_content for d in docs)

        # RAG Chain
        rag_chain = (
            {
                "context": retriever | format_docs,
                "question": RunnablePassthrough()
            }
            | prompt
            | llm
            | StrOutputParser()
        )

        total_pages = len(raw_documents)
        total_chunks = len(chunks)

        return f"""✅ Document processed successfully!
📄 Pages found     : {total_pages}
🧩 Chunks created  : {total_chunks}
🤖 Model ready     : Groq Llama 3.1 8B
🚀 You can now use all features below!"""

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============================================
# FEATURE FUNCTIONS
# ============================================

def chat_fn(question, history):
    global rag_chain
    if rag_chain is None:
        return "⚠️ Please upload and process a document first!"
    if not question.strip():
        return "⚠️ Please type a question!"
    try:
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        return f"❌ Error: {str(e)}"


def summary_fn():
    global llm, raw_documents
    if llm is None or raw_documents is None:
        return "⚠️ Please upload and process a document first!"
    try:
        full_text = " ".join([doc.page_content for doc in raw_documents])
        sample_text = full_text[:3000]

        prompt = f"""
        Summarize these study notes clearly.
        Notes: {sample_text}

        Give:
        1. 📌 Main Topic
        2. 🔑 Key Concepts (bullets)
        3. 📊 Important Points
        4. 🎯 What to Focus On
        """

        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content

    except Exception as e:
        return f"❌ Error: {str(e)}"


def quiz_fn(num_questions):
    global llm, raw_documents
    if llm is None or raw_documents is None:
        return "⚠️ Please upload and process a document first!"
    try:
        full_text = " ".join([doc.page_content for doc in raw_documents])
        sample_text = full_text[:3000]

        prompt = f"""
        Create {int(num_questions)} MCQ questions from these notes.
        Notes: {sample_text}

        Format:
        Q1: [Question]
        A) [Option]
        B) [Option]
        C) [Option]
        D) [Option]
        Correct Answer: [Letter] - [Explanation]
        ---
        """

        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content

    except Exception as e:
        return f"❌ Error: {str(e)}"


def flashcard_fn(num_cards):
    global llm, raw_documents
    if llm is None or raw_documents is None:
        return "⚠️ Please upload and process a document first!"
    try:
        full_text = " ".join([doc.page_content for doc in raw_documents])
        sample_text = full_text[:3000]

        prompt = f"""
        Create {int(num_cards)} flashcards from these notes.
        Notes: {sample_text}

        Format:
        🃏 CARD 1:
        FRONT: [Term or Question]
        BACK: [Answer or Definition]
        ---
        """

        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content

    except Exception as e:
        return f"❌ Error: {str(e)}"


def explain_fn(topic):
    global rag_chain
    if rag_chain is None:
        return "⚠️ Please upload and process a document first!"
    if not topic.strip():
        return "⚠️ Please enter a topic!"
    try:
        question = f"Explain {topic} simply with examples and key points"
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============================================
# BUILD UI
# ============================================
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="🎓 AI Study Assistant"
) as demo:

    # ---- Header ----
    gr.Markdown("""
    # 🎓 AI Study Assistant
    ### Powered by Groq Llama 3.1 + RAG + FAISS + HuggingFace
    Upload your notes and study smarter! 🚀
    """)

    # ==============================
    # DOCUMENT UPLOAD SECTION
    # ==============================
    gr.Markdown("## 📁 Step 1: Upload Your Document")

    with gr.Row():
        with gr.Column(scale=2):
            file_input = gr.File(
                label="📄 Upload PDF",
                file_types=[".pdf"]
            )
            upload_btn = gr.Button(
                "⚡ Process Document",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=3):
            upload_status = gr.Textbox(
                label="📊 Status",
                lines=6,
                placeholder="Upload a PDF and click Process Document...",
                interactive=False
            )

    upload_btn.click(
        fn=process_file,
        inputs=file_input,
        outputs=upload_status
    )

    gr.Markdown("---")
    gr.Markdown("## 🚀 Step 2: Choose a Feature")

    # ==============================
    # FEATURE TABS
    # ==============================
    with gr.Tabs():

        # TAB 1: Chat
        with gr.TabItem("💬 Ask Questions"):
            gr.Markdown("### Ask anything from your uploaded notes!")
            gr.ChatInterface(
                fn=chat_fn,
                examples=[
                    "What are the main topics?",
                    "Summarize the key points",
                    "What are the important concepts?",
                    "What should I focus on?"
                ],
                chatbot=gr.Chatbot(height=400),
                type="messages"
            )

        # TAB 2: Summary
        with gr.TabItem("📋 Summary"):
            gr.Markdown("### Get a quick summary of your notes!")
            summary_btn = gr.Button(
                "📋 Generate Summary",
                variant="primary",
                size="lg"
            )
            summary_output = gr.Textbox(
                label="📋 Summary",
                lines=15,
                placeholder="Click button to generate summary..."
            )
            summary_btn.click(
                fn=summary_fn,
                outputs=summary_output
            )

        # TAB 3: Quiz
        with gr.TabItem("🧪 Quiz Generator"):
            gr.Markdown("### Test your knowledge with MCQs!")
            num_q = gr.Slider(
                minimum=3,
                maximum=10,
                value=5,
                step=1,
                label="Number of Questions"
            )
            quiz_btn = gr.Button(
                "🧪 Generate Quiz",
                variant="primary",
                size="lg"
            )
            quiz_output = gr.Textbox(
                label="🧪 Quiz",
                lines=20,
                placeholder="Click button to generate quiz..."
            )
            quiz_btn.click(
                fn=quiz_fn,
                inputs=num_q,
                outputs=quiz_output
            )

        # TAB 4: Flashcards
        with gr.TabItem("🃏 Flashcards"):
            gr.Markdown("### Quick revision flashcards!")
            num_c = gr.Slider(
                minimum=3,
                maximum=10,
                value=5,
                step=1,
                label="Number of Flashcards"
            )
            flash_btn = gr.Button(
                "🃏 Generate Flashcards",
                variant="primary",
                size="lg"
            )
            flash_output = gr.Textbox(
                label="🃏 Flashcards",
                lines=20,
                placeholder="Click button to generate flashcards..."
            )
            flash_btn.click(
                fn=flashcard_fn,
                inputs=num_c,
                outputs=flash_output
            )

        # TAB 5: Explain Topic
        with gr.TabItem("🎓 Explain Topic"):
            gr.Markdown("### Get simple explanation of any topic!")
            topic_input = gr.Textbox(
                label="Enter Topic",
                placeholder="e.g. Machine Learning, Neural Networks, RTOS..."
            )
            explain_btn = gr.Button(
                "🎓 Explain",
                variant="primary",
                size="lg"
            )
            explain_output = gr.Textbox(
                label="💡 Explanation",
                lines=15,
                placeholder="Enter a topic and click Explain..."
            )
            explain_btn.click(
                fn=explain_fn,
                inputs=topic_input,
                outputs=explain_output
            )

    # ---- Footer ----
    gr.Markdown("""
    ---
    ### 🛠️ Built With:
    `Python` `LangChain` `Groq Llama 3.1` `FAISS` `HuggingFace` `Gradio`
    """)

# Launch
print("🚀 Launching AI Study Assistant...")
demo.launch(share=True, debug=False, quiet=True)

✅ Groq API Key Loaded!


/tmp/ipykernel_3422/2965966293.py:222: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3422/2965966293.py:284: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=400),
/tmp/ipykernel_3422/2965966293.py:284: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(height=400),
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:323: UserWarning: The type of the gr.Chatbot does not matc

🚀 Launching AI Study Assistant...
* Running on public URL: https://809cdef998c89c65dc.gradio.live
